In [1]:
import pandas as pd

# Load the dataset
# Make sure 'train.csv' is in the same directory as your script
df = pd.read_csv(r"C:\Users\HP\Downloads\train.csv (1)\train.csv")

# Check the distribution of our target variable
print(df['target'].value_counts())

target
0    179902
1     20098
Name: count, dtype: int64


In [2]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [3]:
# Look at summary statistics for the first 5 features
print(df[['var_0', 'var_1', 'var_2', 'var_3', 'var_4']].describe())

               var_0          var_1          var_2          var_3  \
count  200000.000000  200000.000000  200000.000000  200000.000000   
mean       10.679914      -1.627622      10.715192       6.796529   
std         3.040051       4.050044       2.640894       2.043319   
min         0.408400     -15.043400       2.117100      -0.040200   
25%         8.453850      -4.740025       8.722475       5.254075   
50%        10.524750      -1.608050      10.580000       6.825000   
75%        12.758200       1.358625      12.516700       8.324100   
max        20.315000      10.376800      19.353000      13.188300   

               var_4  
count  200000.000000  
mean       11.078333  
std         1.623150  
min         5.074800  
25%         9.883175  
50%        11.108250  
75%        12.261125  
max        16.671400  


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop(['ID_code', 'target'], axis=1)
y = df['target']

# Split into training and validation sets (stratify keeps the 9:1 ratio in both sets)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the data (essential for Logistic Regression to perform well)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train baseline model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predict and calculate recall
y_pred = model.predict(X_val_scaled)
print("Baseline Recall:", recall_score(y_val, y_pred))

Baseline Recall: 0.258955223880597


## Feature Engineering

In [5]:
def engineer_features(data):
    # Create a copy so we don't modify the original dataframe directly
    df_feat = data.copy()
    
    # Calculate row-wise statistical aggregates 📊
    df_feat['row_mean'] = df_feat.mean(axis=1)
    df_feat['row_std'] = df_feat.std(axis=1)
    df_feat['row_min'] = df_feat.min(axis=1)
    df_feat['row_max'] = df_feat.max(axis=1)
    df_feat['row_sum'] = df_feat.sum(axis=1)
    
    return df_feat

# Apply the function to our features (X)
X_engineered = engineer_features(X)
print(f"New feature count: {X_engineered.shape[1]}")

New feature count: 205


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Split the data (using our new engineered features)
X_train, X_val, y_train, y_val = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Initialize the scaler
scaler = StandardScaler()

# 3. Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [7]:
from sklearn.linear_model import LogisticRegression

# Initialize with balanced class weights
model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Train the model on our scaled, engineered data
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [8]:
from sklearn.metrics import recall_score, classification_report

# 1. Get the model's predictions on our validation set
y_pred = model.predict(X_val_scaled)

# 2. Calculate and print the recall score
current_recall = recall_score(y_val, y_pred)
print(f"Validation Recall: {current_recall:.4f}")

# 3. Print the full report to see Precision and F1-score as well
print("\nFull Classification Report:")
print(classification_report(y_val, y_pred))

Validation Recall: 0.7796

Full Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87     35980
           1       0.30      0.78      0.43      4020

    accuracy                           0.79     40000
   macro avg       0.63      0.79      0.65     40000
weighted avg       0.90      0.79      0.83     40000



In [9]:
# ==========================================
# Experiment: Transforms + Outliers
# ==========================================

def engineer_features_heavy(data):
    df_feat = data.copy()
    
    # 1. Row-wise stats (Our known good features)
    df_feat['row_mean'] = df_feat.mean(axis=1)
    df_feat['row_std'] = df_feat.std(axis=1)
    df_feat['row_min'] = df_feat.min(axis=1)
    df_feat['row_max'] = df_feat.max(axis=1)
    df_feat['row_sum'] = df_feat.sum(axis=1)
    
    # 2. Power Transforms: Squaring all original features
    for col in data.columns:
        df_feat[f'{col}_squared'] = data[col] ** 2
        
    return df_feat

print("1. Engineering features (Adding squares)...")
X_heavy = engineer_features_heavy(X)

# Split before handling outliers to prevent data leakage
X_train, X_val, y_train, y_val = train_test_split(
    X_heavy, y, test_size=0.2, random_state=42, stratify=y
)

print("2. Handling outliers (Clipping)...")
# Calculate 1st and 99th percentiles using ONLY the training data
lower_bounds = X_train.quantile(0.01)
upper_bounds = X_train.quantile(0.99)

# Clip the extreme values in both sets
X_train_clipped = X_train.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
X_val_clipped = X_val.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

print("3. Scaling...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clipped)
X_val_scaled = scaler.transform(X_val_clipped)

print("4. Training model...")
model = LogisticRegression(class_weight='balanced', penalty='l2', max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate standard threshold (0.5)
y_pred = model.predict(X_val_scaled)
recall = recall_score(y_val, y_pred)
print(f"\nValidation Recall (Transforms + Outliers, Threshold 0.5): {recall:.4f}")

1. Engineering features (Adding squares)...


C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2908955666.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_squared'] = data[col] ** 2
C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2908955666.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_squared'] = data[col] ** 2
C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2908955666.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

2. Handling outliers (Clipping)...
3. Scaling...
4. Training model...

Validation Recall (Transforms + Outliers, Threshold 0.5): 0.7943


In [18]:
# List of C values to test
c_values = [0.001, 0.01, 0.1, 1.0, 10.0]

print("Testing Hyperparameters...\n")

for c_val in c_values:
    # Initialize model with different C values
    test_model = LogisticRegression(
        class_weight='balanced', 
        C=c_val, 
        penalty='l1',     # l1 can act as automatic feature selection
        solver='saga', 
        max_iter=500,     # Lowered slightly for faster testing
        n_jobs=-1         # Use all processor cores
    )
    
    # Train and test
    test_model.fit(X_train_scaled, y_train)
    y_pred_test = test_model.predict(X_val_scaled)
    
    # Calculate recall
    recall = recall_score(y_val, y_pred_test)
    print(f"C={c_val} | Penalty=l1 | Validation Recall: {recall:.4f}")

Testing Hyperparameters...

C=0.001 | Penalty=l1 | Validation Recall: 0.7726
C=0.01 | Penalty=l1 | Validation Recall: 0.7806
C=0.1 | Penalty=l1 | Validation Recall: 0.7803
C=1.0 | Penalty=l1 | Validation Recall: 0.7796
C=10.0 | Penalty=l1 | Validation Recall: 0.7796


In [19]:
# 1. Revert to our cleanest feature set (Original + 5 Row Stats)
def engineer_features(data):
    df_feat = data.copy()
    df_feat['row_mean'] = df_feat.mean(axis=1)
    df_feat['row_std'] = df_feat.std(axis=1)
    df_feat['row_min'] = df_feat.min(axis=1)
    df_feat['row_max'] = df_feat.max(axis=1)
    df_feat['row_sum'] = df_feat.sum(axis=1)
    return df_feat

X_engineered = engineer_features(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42, stratify=y
)

print("Handling outliers...")
# 2. Find the 1st and 99th percentiles for every column in the training data
lower_bounds = X_train.quantile(0.01)
upper_bounds = X_train.quantile(0.99)

# 3. Clip the extreme values so nothing goes beyond those percentiles
X_train_clipped = X_train.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
X_val_clipped = X_val.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

print("Scaling...")
# 4. Scale the newly clipped data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clipped)
X_val_scaled = scaler.transform(X_val_clipped)

print("Training model...")
# 5. Train with our best known settings
model = LogisticRegression(class_weight='balanced', penalty='l2', max_iter=1000)
model.fit(X_train_scaled, y_train)

# Test the recall
y_pred = model.predict(X_val_scaled)
print(f"Validation Recall (Clipped Outliers): {recall_score(y_val, y_pred):.4f}")

Handling outliers...
Scaling...
Training model...
Validation Recall (Clipped Outliers): 0.7796


In [21]:
# 1. Get the raw probabilities for the positive class (class 1)
y_probabilities = model.predict_proba(X_val_scaled)[:, 1]

# 2. Set our new, lower threshold
custom_threshold = 0.40 

# 3. Create new predictions: if probability >= threshold, it's a 1, else 0
y_pred_custom = (y_probabilities >= custom_threshold).astype(int)

# 4. Check our new recall score
new_recall = recall_score(y_val, y_pred_custom)
print(f"Recall with threshold {custom_threshold}: {new_recall:.4f}")

Recall with threshold 0.4: 0.8455


In [23]:
import numpy as np

# Test thresholds from 0.30 to 0.48 in steps of 0.02
thresholds_to_test = np.arange(0.30, 0.50, 0.02)

print("Testing Thresholds:")
for t in thresholds_to_test:
    # Generate predictions for the current threshold
    y_pred_loop = (y_probabilities >= t).astype(int)
    
    # Calculate recall
    loop_recall = recall_score(y_val, y_pred_loop)
    print(f"Threshold: {t:.2f} | Recall: {loop_recall:.4f}")

Testing Thresholds:
Threshold: 0.30 | Recall: 0.9027
Threshold: 0.32 | Recall: 0.8908
Threshold: 0.34 | Recall: 0.8769
Threshold: 0.36 | Recall: 0.8664
Threshold: 0.38 | Recall: 0.8537
Threshold: 0.40 | Recall: 0.8455
Threshold: 0.42 | Recall: 0.8338
Threshold: 0.44 | Recall: 0.8231
Threshold: 0.46 | Recall: 0.8107
Threshold: 0.48 | Recall: 0.7935


In [25]:
# ==========================================
# Experiment: Transforms + Threshold 0.40
# ==========================================

def engineer_features_transforms(data):
    df_feat = data.copy()
    
    # 1. Row-wise stats
    df_feat['row_mean'] = df_feat.mean(axis=1)
    df_feat['row_std'] = df_feat.std(axis=1)
    df_feat['row_min'] = df_feat.min(axis=1)
    df_feat['row_max'] = df_feat.max(axis=1)
    df_feat['row_sum'] = df_feat.sum(axis=1)
    
    # 2. Power Transforms (Squaring)
    for col in data.columns:
        df_feat[f'{col}_squared'] = data[col] ** 2
        
    return df_feat

print("Engineering features (Row Stats + Squares)...")
X_heavy = engineer_features_transforms(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_heavy, y, test_size=0.2, random_state=42, stratify=y
)

print("Scaling...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training model...")
model = LogisticRegression(class_weight='balanced', penalty='l2', max_iter=1000)
model.fit(X_train_scaled, y_train)

print("Evaluating with Threshold 0.40...")
# Get raw probabilities
y_probabilities = model.predict_proba(X_val_scaled)[:, 1]

# Apply custom threshold
test_threshold = 0.40
y_pred_heavy_custom = (y_probabilities >= test_threshold).astype(int)

# Check Recall and Full Report
print(f"\nRecall (Transforms + Threshold 0.40): {recall_score(y_val, y_pred_heavy_custom):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_heavy_custom))

Engineering features (Row Stats + Squares)...


C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2211666996.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_squared'] = data[col] ** 2
C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2211666996.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_squared'] = data[col] ** 2
C:\Users\HP\AppData\Local\Temp\ipykernel_25512\2211666996.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

Scaling...
Training model...
Evaluating with Threshold 0.40...

Recall (Transforms + Threshold 0.40): 0.8515

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.75      0.85     35980
           1       0.27      0.85      0.41      4020

    accuracy                           0.76     40000
   macro avg       0.63      0.80      0.63     40000
weighted avg       0.91      0.76      0.80     40000



In [27]:
# =======================================================
# All-In: Row Stats + Transforms + Custom Threshold
# =======================================================

# 1. Get raw probabilities from your heavy model (the one with squared features)
y_probabilities_heavy = model.predict_proba(X_val_scaled)[:, 1]

# 2. Apply a more aggressive threshold to overcome the feature noise
final_heavy_threshold = 0.34
y_pred_final_heavy = (y_probabilities_heavy >= final_heavy_threshold).astype(int)

# 3. Print the final metrics
final_heavy_recall = recall_score(y_val, y_pred_final_heavy)
print(f"Final Combined Strategy Recall: {final_heavy_recall:.4f}")
print("\nFinal Combined Classification Report:")
print(classification_report(y_val, y_pred_final_heavy))

Final Combined Strategy Recall: 0.8811

Final Combined Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.69      0.81     35980
           1       0.24      0.88      0.38      4020

    accuracy                           0.71     40000
   macro avg       0.61      0.79      0.60     40000
weighted avg       0.91      0.71      0.77     40000



In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, classification_report
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. Load Data
# ==========================================
print("Loading data...")
df = pd.read_csv(r"C:\Users\HP\Downloads\train.csv (1)\train.csv") 

X = df.drop(['ID_code', 'target'], axis=1)
y = df['target']

# ==========================================
# 2. Feature Engineering (Winning Strategy)
# ==========================================
def engineer_features(data):
    print("Engineering row-wise statistical features...")
    df_feat = data.copy()
    
    # Row-wise statistics capture the overall magnitude/variance of a customer's profile
    df_feat['row_mean'] = df_feat.mean(axis=1)
    df_feat['row_std'] = df_feat.std(axis=1)
    df_feat['row_min'] = df_feat.min(axis=1)
    df_feat['row_max'] = df_feat.max(axis=1)
    df_feat['row_sum'] = df_feat.sum(axis=1)
    
    return df_feat

X_engineered = engineer_features(X)

# Split data (stratify ensures 9:1 class imbalance is maintained in both sets)
X_train, X_val, y_train, y_val = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 3. Scaling
# ==========================================
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# ==========================================
# 4. Model Training
# ==========================================
print("Training Logistic Regression...")
# class_weight='balanced' forces the model to heavily penalize missing the rare '1' class
model = LogisticRegression(
    class_weight='balanced', 
    penalty='l2',        
    max_iter=1000
)
model.fit(X_train_scaled, y_train)

# ==========================================
# 5. Evaluation & Threshold Tuning
# ==========================================
print("Evaluating model...")

# Get raw probabilities
y_probabilities = model.predict_proba(X_val_scaled)[:, 1]

# Apply the winning custom threshold
custom_threshold = 0.32
y_pred_custom = (y_probabilities >= custom_threshold).astype(int)

# Final Score
final_recall = recall_score(y_val, y_pred_custom)
print(f"\nTarget Achieved!")
print(f"Final Validation Recall (Threshold {custom_threshold}): {final_recall:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred_custom))

Loading data...
Engineering row-wise statistical features...
Scaling features...
Training Logistic Regression...
Evaluating model...

Target Achieved!
Final Validation Recall (Threshold 0.32): 0.8883

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.62      0.76     35980
           1       0.21      0.89      0.34      4020

    accuracy                           0.65     40000
   macro avg       0.59      0.76      0.55     40000
weighted avg       0.90      0.65      0.72     40000



# Experiment Log: Santander Customer Transaction Prediction

| Experiment | Features Used | Hyperparameters | Scaling | Threshold | Recall Score | Notes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline** | Original 200 features | Default | None | 0.50 | 0.2589 | Standard logistic regression fails to find minority class due to severe 9:1 class imbalance. |
| **Exp 1** | Original + Row Stats (mean, std, min, max, sum) | `class_weight='balanced'` | StandardScaler | 0.50 | 0.7796 | Huge jump! Adding row statistics and balanced weights heavily penalized missing the rare '1' class. |
| **Exp 2** | Original + Row Stats | `class_weight='balanced'` | StandardScaler | 0.40 | 0.8455 | Lowered the prediction threshold to force the model to be more aggressive on positive predictions. |
| **Exp 3** | Original + Row Stats + Squared Transforms | `class_weight='balanced'` | StandardScaler with Outlier Clipping (1st/99th) | 0.50 | 0.7943 | Handled outliers and added power transforms (squaring all features). The extra 200 columns introduced noise, limiting gains. |
| **Exp 4** | Original + Row Stats + Squared Transforms | `class_weight='balanced'` | StandardScaler | 0.40 | ~0.8500 | Combined the heavy feature set (squares) with an aggressive 0.40 threshold to overcome the noise. |
| **Exp 5 (Clean Strategy)** | Original + Row Stats | `class_weight='balanced'` | StandardScaler | 0.32 | 0.8883 | **Target Achieved!** Found the optimal threshold for the clean feature set. |
| **Final (Heavy Strategy)** | Original + Row Stats + Squared Transforms | `class_weight='balanced'` | StandardScaler | 0.34 | **0.8811** | **Target Achieved!** Combined heavy feature engineering (transforms) with a 0.34 threshold to successfully surpass the 0.88 requirement. |